# C3 · Extracción óptima (2 variantes → 2 métodos)

**Spec:** [`docs/spec_C3_codex_optimal_extraction.md`](../docs/spec_C3_codex_optimal_extraction.md)  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs12b_realigned`

Extracción óptima (Horne) ponderada por la PSF, en **dos variantes obligatorias** que se diferencian solo en el fondo que se resta antes: `optimal_ls` (superficie local de 04b, comparable 1:1 con C2) y `optimal_psfsub` (modelo de PSF de la primaria, de C1). Son **2 de los 6 métodos** de la cadena: 5 etapas, 6 métodos.

| | |
|---|---|
| **Entrada** | Cubo + PSF (C1) |
| **Salida (QC/productos)** | `stages/spec_optimal_qc.json`, `spec_optimal_object.fits` (ls), `spec_optimal_psfsub_object.fits` |
| **Consume aguas abajo** | D1, E1 |


## Qué hace C3 y las dos variantes

C3 es **extracción óptima de Horne (1986)**: por canal, pondera cada píxel por el **perfil de PSF esperado** (de C1) y la varianza inversa (`f = Σ M·P·D/V / Σ M·P²/V`). Al bajar el peso de los píxeles ruidosos, **gana S/N** frente a la apertura (aquí ~**6.9× mediana**). La fórmula es cerrada; el valor está en implementarla exacta (tests analíticos de flujo y varianza).

**Dos variantes del fondo** — mismo estimador, distinto fondo restado antes:
- **`optimal_ls`** — usa el residual de superficie local (04b) como fondo. Mismo fondo que C2, así que la comparación con C2 aísla la ganancia del ponderado óptimo.
- **`optimal_psfsub`** — ajusta y **resta la PSF de la primaria** primero, y luego extrae ópticamente el compañero. Anticipa el fondo que usará C4, así que `ls` vs `psfsub` es un **diagnóstico del modelo de halo** para D1, no una redundancia.

> **De dónde salen los 6 métodos.** C2–C6 son **cinco etapas**, pero C3 emite estas **dos** variantes como productos separados, así que la cadena compara **seis** métodos: `aperture`, `optimal_ls`, `optimal_psfsub`, `psffit`, `sgf` y `lpm` (el `METHOD_ORDER` que usan D1, D2, E4 y G1).

**Decisión:** G1 **valida `psfsub`** (`validated_with_bias`) y la usa como una de las dos citables (con psffit). **`ls` sobre-sustrae el continuo** (el pedestal de 04b) → sesgo de continuo **−373 % vs apertura**, `v3_continuum_bias` **falla**. Ambas comparten la forma roja real (SED de enana fría) pero `ls` queda con un gran desplazamiento negativo.

Errores **empíricos** (M5 rojo); **robusta a errores de PSF** (±10 % FWHM → 0 % de sesgo de flujo).

> **Nota (revisión 2026-07-11):** el continuo de psfsub sale **negativo** — sobre-sustracción. La sección *Diagnóstico* de abajo muestra que es un **residuo del halo AO cromático** (peor en el azul), **emparejado en los controles**, correctable con un *annulus background* que el código soporta pero que psfsub/D2 no aplican. No afecta el límite de Hα; sí el nivel absoluto para caracterización.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x02_optimal.sh --run-id $RUN
```

Moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/spec_optimal_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x02_optimal.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/spec_optimal_qc.json', RUN_ID)
nb.show(qc, keys=['variants', 'snr_gain_vs_aperture.median', 'continuum_bias_vs_aperture_pct', 'v3_continuum_bias_ok', 'fwhm_pm10pct'], title='C3')


## Los chequeos del QC, en físico

| Chequeo | ¿Qué pregunta contesta? | Si falla |
|---|---|---|
| `v1_snr_gain_ok` | **¿Pesar por la PSF gana algo frente a sumar en una caja?** Mediana de S/N(óptima)/S/N(apertura 3×3) ≥ 1 (se espera 1.1–1.3). | El método no aporta: la extracción óptima solo se justifica por la ganancia de S/N. |
| `v2_error_ratio_ok` | **¿La barra de error describe el ruido real?** Igual que en C2: propagado vs empírico de controles, razón mediana en [0.7, 1.4]. | El σ no es el ruido y la significancia posterior queda mal escalada. |
| `v3_continuum_bias_ok` | **¿El método se come el continuo del compañero?** (óptima − apertura)/apertura en bandas de continuo, < 2–3%. | Es el chequeo que **rechaza `optimal_ls`** en esta cadena: sobre-sustrae el halo y deja el continuo negativo. La variante que G1 valida es `optimal_psfsub`. |
| `v4_clip_concentration_ok` | **¿El rechazo de píxeles se está comiendo la señal?** El mapa de clipping no debe concentrarse en la posición del compañero (< 2× la tasa media). | Se estaría recortando el objeto y llamándolo ruido. |
| `v5_ls_vs_psfsub_written` | **¿Quedan las dos variantes en disco para compararlas?** No juzga: entrega el diagnóstico del modelo de halo a D1. | Falta insumo para D1. |


## Resultados que llevaron a la conclusión

Ganancia de S/N, sesgo de continuo, modelo psfsub y chequeos del `spec_optimal_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('C3', 'stages/spec_optimal_qc.json'):
        q = nb.load_qc('stages/spec_optimal_qc.json', RUN_ID)
        sg = q['snr_gain_vs_aperture']; ck = q['checks']; pm = q['psfsub_model']
        print('variantes:', q['variants'], '| ventana:', q['window_px'], 'px | PSF:', q['psf_model'].split('/')[-1])
        print(f"ganancia S/N vs apertura: mediana {sg['median']:.2f}× (p10 {sg['p10']:.2f}, p90 {sg['p90']:.1f})")
        print(f"sesgo de continuo vs apertura: {q['continuum_bias_vs_aperture_pct']:.0f}%   <-- ls sobre-sustrae")
        print(f"sensibilidad a PSF (±10% FWHM): {q['psf_sensitivity']['fwhm_pm10pct_flux_bias_pct']:.1f}% de sesgo (robusto)")
        print(f"psfsub: amplitud mediana {pm['amplitude_median']:.3g}, n_fit {pm['n_fit_median']:.0f}, "
              f"fit_radius {pm['fit_radius_px']:.0f}px, exclude {pm['exclude_radius_px']:.0f}px")
        print(f"checks: v1_snr_gain={ck['v1_snr_gain_ok']} v2_error={ck['v2_error_ratio_ok']} "
              f"v3_continuum_bias={ck['v3_continuum_bias_ok']} (falla: sesgo ls) v4_clip={ck['v4_clip_concentration_ok']}")
        print('open_issue:', q['open_issues'][0])


## Plot — las dos variantes: ls vs psfsub

Los espectros suavizados de `spec_optimal_object.fits` (ls) y `spec_optimal_psfsub_object.fits` (psfsub). **`ls` (naranja)** queda muy por debajo de cero = sobre-sustracción del continuo (el sesgo −373 %); **`psfsub` (verde)** queda cerca de cero y **sube al rojo** (SED real de enana fría). Comparten la forma, difieren en nivel — por eso G1 valida psfsub y rechaza ls.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    bias_pct = nb.load_qc('stages/spec_optimal_qc.json', RUN_ID)['continuum_bias_vs_aperture_pct']
    def spec(path):
        h = fits.open(rd / 'stages' / path); d = h[1].data
        w = np.asarray(d['wave_A'], float); f = np.asarray(d['flux'], float); h.close(); return w, f
    from musepipe.spectral import median_filter_1d
    sm = lambda x, n=41: median_filter_1d(x, n)   # mediana móvil, ignora NaN
    w, fls = spec('spec_optimal_object.fits')
    _, fps = spec('spec_optimal_psfsub_object.fits')
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(w, sm(fls), lw=1.2, color='tab:orange', label='optimal_ls (superficie local)')
    ax.plot(w, sm(fps), lw=1.2, color='tab:green', label='optimal_psfsub (resta de PSF) — validada G1')
    ax.axhline(0, color='0.6', lw=0.7); ax.axvline(6563, color='tab:red', ls=':', label='Hα')
    allv = np.concatenate([sm(fls), sm(fps)])
    ax.set_ylim(np.nanpercentile(allv, 2), np.nanpercentile(allv, 98))
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo (suavizado 41ch)')
    ax.set_title(f'C3 · dos variantes: ls sobre-sustrae el continuo ({bias_pct:.0f}% vs apertura), psfsub no')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'c3_optimal'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'ls_vs_psfsub.png', dpi=110); print('figura ->', outdir / 'ls_vs_psfsub.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Diagnóstico — ¿sobre-sustracción del continuo? (objeto vs controles)

El continuo de psfsub sale **negativo**. La prueba clave: comparar el objeto con la **media de los 33 controles** (fondo/ruido puro, debería ~0). Hallazgo:

- La media de controles **no** está en 0 y es **cromática**: ~−1315 (azul) → −210 (rojo). Es el **residuo del halo AO** que la resta de PSF deja (peor en el azul, donde el halo es más ancho — encaja con C1/r0).
- **`objeto − controles`** recupera el **continuo físico** del compañero: sube al rojo (SED de enana fría), cerca de 0 en el azul, **nada en Hα**.
- El código soporta este *annulus background* (`optimal.py: local_bkg_annulus_px`) y **C2 sí lo aplica** (sus controles quedan en ~0); psfsub y el calibrado D2 **no** → el continuo entregado queda sesgado. **No afecta el límite de Hα** (fondo suave, y la detección usa control=objeto); **sí** sesga el nivel absoluto para caracterización (G3).


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    h = fits.open(rd / 'stages' / 'spec_optimal_psfsub_object.fits')
    wave = np.asarray(h[1].data['wave_A'], float)   # eje λ del propio producto
    fo = np.asarray(h[1].data['flux'], float); h.close()
    C = np.load(rd / 'stages' / 'spec_optimal_psfsub_controls.npz')['control_spectra']
    cm = np.nanmean(C, axis=0)
    print('banda            media_ctrl     objeto   obj-ctrl   (unidades nativas)')
    for lo, hi in [(5100, 5500), (6600, 7200), (8000, 8800)]:
        b = (wave >= lo) & (wave <= hi)
        print(f'  {lo}-{hi} Å   {np.nanmedian(cm[b]):9.0f}  {np.nanmedian(fo[b]):9.0f}  {np.nanmedian((fo - cm)[b]):9.0f}')
    from musepipe.spectral import median_filter_1d
    sm = lambda x, n=81: median_filter_1d(x, n)   # mediana móvil, ignora NaN
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.plot(wave, sm(fo), lw=1, color='tab:green', label='objeto psfsub (crudo, sobre-sustraído)')
    ax.plot(wave, sm(cm), lw=1, color='tab:red', ls='--', label=f'media de {C.shape[0]} controles = fondo residual del halo')
    ax.plot(wave, sm(fo - cm), lw=1.6, color='k', label='objeto − controles = continuo físico')
    ax.axhline(0, color='0.6', lw=0.7); ax.axvline(6563, color='tab:red', ls=':', label='Hα')
    allv = np.concatenate([sm(fo), sm(cm), sm(fo - cm)])
    ax.set_ylim(np.nanpercentile(allv, 2), np.nanpercentile(allv, 98))
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo (suavizado 81ch)')
    ax.set_title('C3 diagnóstico: la media de controles = sobre-sustracción cromática del halo; '
                 'restarla recupera el continuo físico')
    ax.legend(fontsize=8, loc='lower right'); fig.tight_layout()
    outdir = rd / 'plots' / 'c3_optimal'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'oversubtraction_diag.png', dpi=110)
    print('figura ->', outdir / 'oversubtraction_diag.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Figura de paper — el espectro sin binar, con su error y sus líneas

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    from astropy.io import fits
    ROOT_P = nb.project_root()
    METHOD_P = 'optimal_ls'
    PRODUCT_P = 'spec_optimal_object.fits'
    TARGET_P = (nb.run_target(RUN_ID) or RUN_ID).replace(' ', '')
    _h = fits.open(nb.run_dir(RUN_ID) / 'stages' / PRODUCT_P)
    _d = _h[1].data
    _cols = list(_d.columns.names)
    # La unidad viaja con el dato (BUNIT); no hay default silencioso.
    BUNIT_P = _h[1].header.get('BUNIT') or 'ADU'
    W_P = np.asarray(_d['wave_A'], float)
    F_P = np.asarray(_d['flux'], float)
    # El empírico manda; `flux_err` es el que eligió la etapa y solo
    # aporta algo cuando NO es el empírico (ver la nota de abajo).
    E_P = np.asarray(_d['flux_err_emp' if 'flux_err_emp' in _cols
                        else 'flux_err'], float)
    E_ALT_P = np.asarray(_d['flux_err'], float)
    EXTRA_P = {'flux_err_stat': E_ALT_P}
    for _c in ('apcorr', 'npix_eff', 'flags'):
        if _c in _cols:
            EXTRA_P[_c] = np.asarray(_d[_c])
    _h.close()
    try:
        MODO_P = (nb.load_qc('stages/spec_optimal_qc.json', RUN_ID).get('errors') or {}).get('mode')
    except Exception:
        MODO_P = None
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'espectro del compañero · optimal_ls (C3)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c3_optimal'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper_ls' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper_ls' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper_ls' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Figura de paper — el espectro sin binar, con su error y sus líneas

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    from astropy.io import fits
    ROOT_P = nb.project_root()
    METHOD_P = 'optimal_psfsub'
    PRODUCT_P = 'spec_optimal_psfsub_object.fits'
    TARGET_P = (nb.run_target(RUN_ID) or RUN_ID).replace(' ', '')
    _h = fits.open(nb.run_dir(RUN_ID) / 'stages' / PRODUCT_P)
    _d = _h[1].data
    _cols = list(_d.columns.names)
    # La unidad viaja con el dato (BUNIT); no hay default silencioso.
    BUNIT_P = _h[1].header.get('BUNIT') or 'ADU'
    W_P = np.asarray(_d['wave_A'], float)
    F_P = np.asarray(_d['flux'], float)
    # El empírico manda; `flux_err` es el que eligió la etapa y solo
    # aporta algo cuando NO es el empírico (ver la nota de abajo).
    E_P = np.asarray(_d['flux_err_emp' if 'flux_err_emp' in _cols
                        else 'flux_err'], float)
    E_ALT_P = np.asarray(_d['flux_err'], float)
    EXTRA_P = {'flux_err_stat': E_ALT_P}
    for _c in ('apcorr', 'npix_eff', 'flags'):
        if _c in _cols:
            EXTRA_P[_c] = np.asarray(_d[_c])
    _h.close()
    try:
        MODO_P = (nb.load_qc('stages/spec_optimal_qc.json', RUN_ID).get('errors') or {}).get('mode')
    except Exception:
        MODO_P = None
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'espectro del compañero · optimal_psfsub (C3)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c3_optimal'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper_psfsub' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper_psfsub' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper_psfsub' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- Extracción óptima de Horne ponderada por la PSF de C1 → **~6.9× ganancia de S/N** vs apertura (`v1` pasa).
- Dos variantes: **`optimal_psfsub` validada por G1** (`validated_with_bias`) y **`optimal_ls` rechazada** — sobre-sustrae el continuo (sesgo −373%, `v3` falla).
- Errores empíricos (M5 rojo); robusta a errores de PSF (±10% FWHM → 0% de sesgo de flujo).


## Conclusión (registrada)

**C3: extracción óptima de Horne; ~6.9× ganancia de S/N vs apertura; dos variantes (ls, psfsub).**

- **Fecha:** cadena D1 v2 sobre el run realineado (2026-07-09).
- **Entrada:** cubo stage02 + PSF de C1 (`psf_model.json`); ventana 8 px.
- **psfsub** (resta de PSF de la primaria): validada por G1, una de las dos citables.
- **ls** (superficie local): **sobre-sustrae** el continuo (−373 % vs apertura, `v3_continuum_bias` falla) — rechazada.
- **Robusta a PSF** (±10 % FWHM → 0 % de sesgo); errores empíricos (M5 rojo).
- **Hallazgo (2026-07-11):** el continuo de psfsub es negativo por un **residuo de halo AO cromático** (media de controles −1315 azul → −210 rojo), emparejado en controles. `objeto − controles` recupera el continuo físico (sube al rojo). Correctable con annulus background (soportado por el código, aplicado en C2, NO en psfsub/D2). No afecta Hα; sí el nivel absoluto para G3. Ver [[c3-continuum-oversubtraction]].
- **Downstream:** psfsub entra en D1 (par primario psffit vs optimal_psfsub); ls no.
